# 실험 제목: 토큰(Token) 계산과 비용 제어

**목표**: LLM이 텍스트를 처리하는 단위인 '토큰'의 개념을 이해하고, `tiktoken`을 사용해 텍스트의 토큰 수를 직접 계산해봅니다. 또한 `max_tokens` 파라미터로 응답 길이를 제어해봅니다.

In [ ]:
# 필요한 라이브러리 import
import os
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# .env 파일 로드 및 OpenAI 클라이언트 생성
load_dotenv()
client = OpenAI()

### 1. tiktoken으로 토큰 수 계산하기
영어와 한국어의 토큰 소비량이 어떻게 다른지 확인해 봅니다.

In [ ]:
# gpt-4o 계열 모델이 사용하는 인코딩 방식 가져오기
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

text_en = "Hello, how are you today?"
text_ko = "안녕하세요, 오늘 하루 어떠신가요?"

tokens_en = encoding.encode(text_en)
tokens_ko = encoding.encode(text_ko)

print(f"영어 문장 글자수: {len(text_en)} -> 토큰 수: {len(tokens_en)}")
print(f"한국어 문장 글자수: {len(text_ko)} -> 토큰 수: {len(tokens_ko)}")
print("\n(일반적으로 영어는 1단어=1토큰에 가깝지만, 한국어는 형태소나 글자 단위로 더 잘게 쪼개져 토큰 소모가 더 큽니다.)")

### 2. max_tokens 설정으로 출력 제어하기
`max_tokens`는 모델이 생성할 수 있는 **최대 토큰 수**를 제한합니다. 비용을 관리하거나 응답 길이를 강제로 자를 때 사용합니다.

In [ ]:
prompt = "한국의 역사에 대해 설명해줘."

# max_tokens를 20으로 아주 작게 설정
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=20  # 생성되는 토큰 수를 최대 20개로 제한
)

output_text = response.choices[0].message.content
finish_reason = response.choices[0].finish_reason

print("[응답 결과]")
print(output_text)
print("\n[종료 사유 (finish_reason)]:", finish_reason)
print("(finish_reason이 'length'이면 max_tokens에 도달하여 문장이 중간에 잘렸음을 의미합니다.)")

### 실험 결과 정리

* **토큰 인코딩**: 동일한 의미의 문장이라도 영어보다 한국어 데이터가 토큰을 더 많이 소모한다는 것을 `tiktoken` 실습으로 확인했습니다.
* **max_tokens 제한**: `max_tokens` 파라 파라미터를 작게 설정하면 응답이 강제로 잘리고, `finish_reason`이 `length`로 반환됨을 확인했습니다.

### Notion 실험 로그 기록용

* **결과**: `tiktoken`을 활용한 한/영 토큰 소비량 비교 및 `max_tokens` 설정을 통한 생성 길이 제한 테스트를 성공적으로 완료함.
* **문제점**: RAG나 요약 작업에서 한국어 문서를 다룰 때 토큰 제한(Context Window)에 더 빨리 도달할 수 있어 세밀한 Chunk 분할이 필요함을 인지함.
* **다음 개선 방향**: 이후 LangChain의 Text Splitter 부분에서 토큰 수를 기준으로 텍스트를 나누는 방법(TokenTextSplitter)을 적용해 볼 예정.